# 03 — Caching, Benchmarking, and Visualization

This chapter demonstrates performance engineering for local LLM apps.


## 1) Benchmark Objective

Evaluate 4 local models on:
- latency
- throughput
- memory proxy
- lightweight output quality proxy

**Best practice**: benchmark repeated runs per model under same prompt settings.


In [ ]:
from streamlit_app.config import APP_CONFIG
from streamlit_app.utils.models import is_ollama_available, run_benchmark_matrix

ollama_up = is_ollama_available()
models = APP_CONFIG.models.benchmark_models
prompt = 'Explain why caching improves user experience in AI applications.'

if ollama_up:
    summary_rows, run_rows = run_benchmark_matrix(models=models, prompt=prompt, runs=2, system_prompt=None, temperature=0.2)
else:
    summary_rows, run_rows = [], []

{'ollama_available': ollama_up, 'models_requested': models, 'summary_rows': len(summary_rows), 'run_rows': len(run_rows)}


## 2) Create Benchmark DataFrames

If Ollama is not available, this notebook still executes and clearly reports why live benchmarks were skipped.


In [ ]:
import pandas as pd

summary_df = pd.DataFrame(summary_rows)
runs_df = pd.DataFrame(run_rows)

if summary_df.empty:
    summary_df = pd.DataFrame(columns=['model', 'mean_latency', 'mean_throughput_wps', 'mean_memory_mb', 'mean_quality_score'])
if runs_df.empty:
    runs_df = pd.DataFrame(columns=['model', 'latency_seconds', 'output_word_count'])

summary_df.head(), runs_df.head()


## 3) Visualization

**Visual explanation**:
- bar chart for model-level latency
- box plot for run-level latency spread
- throughput chart for speed comparison

**Common mistake**: plotting only averages and hiding variance.


In [ ]:
from pathlib import Path
import plotly.express as px

fig_dir = Path('outputs/figures')
fig_dir.mkdir(parents=True, exist_ok=True)

if not summary_df.empty:
    latency_fig = px.bar(summary_df, x='model', y='mean_latency', color='model', title='Mean latency by model')
    latency_fig.write_html(str(fig_dir / 'nb03_mean_latency.html'), include_plotlyjs='cdn')

    throughput_fig = px.bar(summary_df, x='model', y='mean_throughput_wps', color='model', title='Mean throughput by model')
    throughput_fig.write_html(str(fig_dir / 'nb03_mean_throughput.html'), include_plotlyjs='cdn')

{'figures_dir': str(fig_dir), 'latency_chart_written': not summary_df.empty}


## 4) Cache Impact Demonstration

This compares uncached and cached sentiment inference timing.


In [ ]:
from streamlit_app.utils.caching import compare_cached_vs_uncached_sentiment

if ollama_up:
    cache_metrics = compare_cached_vs_uncached_sentiment(
        'This app feels fast and easy to use.',
        APP_CONFIG.models.sentiment,
    )
else:
    cache_metrics = {
        'uncached_seconds': None,
        'cached_first_seconds': None,
        'cached_second_seconds': None,
        'note': 'Skipped because Ollama daemon is unavailable.'
    }

cache_metrics


## 5) Export Benchmark Metrics

Artifacts are stored in `outputs/metrics` for reproducibility and portfolio evidence.


In [ ]:
from pathlib import Path

metrics_dir = Path('outputs/metrics')
metrics_dir.mkdir(parents=True, exist_ok=True)

summary_path = metrics_dir / 'nb03_summary.csv'
runs_path = metrics_dir / 'nb03_runs.csv'

summary_df.to_csv(summary_path, index=False)
runs_df.to_csv(runs_path, index=False)

{'summary_path': str(summary_path), 'runs_path': str(runs_path), 'rows_saved': len(runs_df)}
